In [2]:
import pandas as pd

df = pd.read_csv("data/contributions.csv", dtype=str)

# mask for rows with missing first/last and not CITYPAC
mask_missing = (df['contributor_first_name'].isna() | df['contributor_last_name'].isna())
mask_non_pac = ~df['contributor_name'].str.contains("CITYPAC", case=False, na=False)

rows_to_fix = df[mask_missing & mask_non_pac]


In [3]:
# Exemple:  Count rows where last_name is exactly "LARSON FOR CONGRESS"
count_larson = df[df['contributor_last_name'] == "LARSON FOR CONGRESS"].shape[0]

print("Number of rows where last_name is 'LARSON FOR CONGRESS':", count_larson)


Number of rows where last_name is 'LARSON FOR CONGRESS': 27


In [4]:
row_1 = rows_to_fix.iloc[0]
print(row_1['contributor_name'].strip())

LARSON FOR CONGRESS


In [5]:
rows_to_fix

,contributor_name,contributor_first_name,contributor_last_name,contributor_street_1,contributor_street_2,contributor_city,contributor_state,contributor_zip,contributor_employer,contributor_occupation,...,is_refunded_or_reattributed,contribution_refunded_amount,sub_id,transaction_id,two_year_transaction_period,lat,lng,geocode_provider,geocode_checked_at,contributor_year
431,LARSON FOR CONGRESS,NaN,NaN,PO BOX 261172,NaN,HARTFORD,CT,06126,NaN,NaN,...,NaN,NaN,4123020221645492339,A1C64B7439C514E7BAB0,2022,41.764582,-72.6908547,nominatim,2025-08-15 20:17:44.185238-07,2022
649,JAIME FOR CONGRESS,NaN,NaN,PO BOX 1614,NaN,RIDGEFIELD,WA,986420020,NaN,NaN,...,NaN,NaN,4010320231645641475,AD8CDD488677841C4B8E,2022,45.8167616,-122.7433775,nominatim,2025-08-16 00:40:57.452992-07,2022
1065,GRAND CANYON STATE CAUCUS,NaN,NaN,7108 N 20TH ST,NaN,PHOENIX,AZ,850205648,NaN,NaN,...,NaN,NaN,4032520241885656503,AC6E0BAA3E0314D168BB,2024,33.5405585,-112.0393149,nominatim,2025-08-17 03:44:49.597806-07,2024
1740,NRCC,NaN,NaN,320 FIRST STREET SE,.,WASHINGTON,DC,20003,NaN,NaN,...,NaN,NaN,4011420251130092928,AC957464A69CB40988C3,2024,38.8716043,-77.0057659,nominatim,2025-08-15 06:51:11.148931-07,2024
1929,DSCC,NaN,NaN,120 MARYLAND AVE NE,NaN,WASHINGTON,DC,20002,NaN,NaN,...,NaN,NaN,4072420241978933190,ABAADD933CC624D17B6C,2024,38.8916878,-77.0049145,nominatim,2025-08-17 06:16:44.640409-07,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158259,HEARTLAND PAC,NaN,NaN,25330 LETCHWORTH ROAD,NaN,BEACHWOOD,OH,441224159,NaN,NaN,...,NaN,NaN,4122920221645228609,ABDBFD94A7E80448883A,2022,41.4717853,-81.5001073,nominatim,2025-08-17 04:54:23.570932-07,2022
158341,HEARTLAND PAC,NaN,NaN,25330 LETCHWORTH ROAD,NaN,BEACHWOOD,OH,441224159,NaN,NaN,...,NaN,NaN,4011420231698187046,A5FE81F2A1E7948758A7,2022,41.4717853,-81.5001073,nominatim,2025-08-17 05:01:42.809816-07,2022
158866,DSCC,NaN,NaN,120 MARYLAND AVE NE,NaN,WASHINGTON,DC,20002,NaN,NaN,...,NaN,NaN,4062320251206919007,A6113E3C982464D92AA5,2026,38.8916878,-77.0049145,nominatim,2025-08-17 05:22:46.017649-07,2025
159086,GROW THE MAJORITY,NaN,NaN,228 S WASHINGTON ST STE 115,NaN,ALEXANDRIA,VA,22314,NaN,NaN,...,NaN,NaN,4032020251180458816,AEA44A4E1E81D46B4B42,2026,38.8005028,-77.0479401,nominatim,2025-08-17 05:34:18.397164-07,2025


In [6]:
def normalize_zip(zip_code):
    """Return the first 5 digits of a ZIP code (handle ZIP+4)."""
    if pd.isna(zip_code):
        return ""
    z = str(zip_code).strip()
    return z[:5]  # only keep first 5 digits


In [7]:
fixed_rows = []   
unmatched_rows = []  

# Precompute normalized ZIP
df['zip5'] = df['contributor_zip'].apply(normalize_zip)
rows_to_fix['zip5'] = rows_to_fix['contributor_zip'].apply(normalize_zip)
for idx, row in rows_to_fix.iterrows():
    base_name = row['contributor_name'].strip()
    zip5 = row['zip5']

    # candidates: same ZIP5, name starts with base_name, and have valid names
    candidates = df[
        (df['zip5'] == zip5) &
        (df['contributor_name'].str.startswith(base_name)) &
        df['contributor_first_name'].notna() &
        df['contributor_last_name'].notna()
    ]

    if not candidates.empty:
        match = candidates.iloc[0]

        # fill missing values
        df.at[idx, 'contributor_first_name'] = match['contributor_first_name']
        df.at[idx, 'contributor_last_name'] = match['contributor_last_name']
        df.at[idx, 'contributor_name'] = match['contributor_name']

        fixed_rows.append({
            "index": idx,
            "old_contributor_name": row['contributor_name'],
            "new_contributor_name": match['contributor_name'],
            "first_name_added": match['contributor_first_name'],
            "last_name_added": match['contributor_last_name'],
            "zip_original": row['contributor_zip'],
            "zip_match": match['contributor_zip']
        })
    else:
        unmatched_rows.append(row)


C:\Users\mehya\AppData\Local\Temp\ipykernel_23964\435557161.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rows_to_fix['zip5'] = rows_to_fix['contributor_zip'].apply(normalize_zip)


In [24]:
fixed_df = pd.DataFrame(fixed_rows)
#fixed_df.to_csv("fixed_missing_names.csv", index=False)

if unmatched_rows:
    unmatched_df = pd.DataFrame(unmatched_rows)
    unmatched_df.to_csv("contributions_contributors.csv", index=False)
    print(f"Saved {len(unmatched_rows)} unmatched rows to unmatched_missing_names.csv")
else:
    print("All missing names were matched and filled.")

Saved 335 unmatched rows to unmatched_missing_names.csv


In [19]:
alldata = pd.read_csv("data/contributions.csv", dtype=str)

unmatched_names = set(unmatched_df['contributor_name'])
data_indiv = alldata[~alldata['contributor_name'].isin(unmatched_names)].copy()


In [25]:
data_indiv.to_csv("contributions_individuals.csv", index=False)

In [23]:
unique_indiv = data_indiv['contributor_name'].nunique()
print("Number of unique Indiv names:", unique_indiv)

Number of unique Indiv names: 16382


In [10]:
unique_contributors = unmatched_df['contributor_name'].nunique()

print("Number of unique contributor names:", unique_contributors)

Number of unique contributor names: 34


## Checking Data

In [11]:
unique_per_cycle = unmatched_df.groupby('two_year_transaction_period')['contributor_name'].nunique().reset_index()

unique_per_cycle.rename(columns={'contributor_name': 'unique_contributors'}, inplace=True)

print(unique_per_cycle)

  two_year_transaction_period  unique_contributors
0                        2022                   14
1                        2024                   25
2                        2026                   11


In [ ]:
unmatched_df['contribution_receipt_amount'] = pd.to_numeric(unmatched_df['contribution_receipt_amount'], errors='coerce')

# Group by contributor_name and sum contributions
total_per_contributor = unmatched_df.groupby('contributor_name')['contribution_receipt_amount'].sum().reset_index()

# Rename column
total_per_contributor.rename(columns={'contribution_receipt_amount': 'total_contributed'}, inplace=True)

# Sort descending to see top contributors
total_per_contributor = total_per_contributor.sort_values(by='total_contributed', ascending=False)


In [13]:
total_per_contributor

,contributor_name,total_contributed
19,MARYLAND ASSOCIATION FOR CONCERNED CITIZENS PO...,149900.00
32,TO PROTECT OUR HERITAGE PAC,146050.00
30,SUN POLITICAL ACTION COMMITTEE (SUNPAC),54200.00
6,"DEMOCRACY ENGINE, LLC",47447.20
13,HEARTLAND PAC,42225.96
18,LOUISIANIANS FOR AMERICAN SECURITY POLITICAL A...,37800.00
11,GRAND CANYON STATE CAUCUS,37000.00
20,MID MANHATTAN POLITICAL ACTION COMMITTEE (MID ...,24000.00
7,DSCC,18496.00
8,FRIENDS OF ISRAEL POLITICAL ACTION COMMITTEE -...,17000.00


In [14]:
df['contribution_receipt_amount'] = pd.to_numeric(df['contribution_receipt_amount'], errors='coerce')

# Group by contributor_name and sum contributions
total_per_contributor = df.groupby('contributor_name')['contribution_receipt_amount'].sum().reset_index()

# Rename column for clarity
total_per_contributor.rename(columns={'contribution_receipt_amount': 'total_contributed'}, inplace=True)

# Sort descending to see top contributors
total_per_contributor = total_per_contributor.sort_values(by='total_contributed', ascending=False)

  # show top 20 contributors

In [15]:
total_per_contributor

,contributor_name,total_contributed
12881,"SCHLEIFER, HARRIET",563700.00
8793,"LEVY, EDWARD",480000.00
16196,"ZALIK, DAVID",453800.00
16197,"ZALIK, HELEN",453800.00
10851,"PANAR, MANNY",416700.00
...,...,...
2492,"COMER FOR CONGRESS, JAMES REP. JR.",-4127.74
9900,"MIKE KELLY FOR CONGRESS, MIKE REP. JR.",-4683.70
5833,"GUTHRIE FOR CONGRESS, BRETT REP.",-4709.89
2245,CLIFF BENTZ FOR CONGRESS,-5000.00
